# 06 · Who added attacking value? Expected Threat

**Question:** who added the most attacking value through passing and carrying in a game, and across the 2022 World Cup?

This notebook fits a location-based Expected Threat (xT) surface on **2018 World Cup events only** and applies the frozen surface to **2022 events**. It creates action-level explanations, player-match and tournament tables, pitch maps, and provenance reports. It uses the corrected minutes method without requiring other notebooks in the same kernel.

xT is not total player impact. It misses off-ball movement, defensive positioning, pass difficulty, score state and many tactical effects. It is not a causal estimate, recruitment prediction, or financial valuation. Shooting statistics remain alongside movement xT; we do not add them into an arbitrary overall score.

Method references: [Karun Singh's xT introduction](https://karun.in/blog/expected-threat.html), [socceraction documentation](https://socceraction.readthedocs.io/en/latest/documentation/valuing_actions/xT.html). Our explicit variant includes failed pass attempts as zero-valued terminal transitions during fitting; it rates **successful observed movements only**. It is not an exact reproduction of the reference implementation.

Data: **StatsBomb Open Data**. Follow the supplied README/license and attribution/logo requirements before publishing. This notebook keeps provider data local; Metrica is not joined to these named players.


In [ ]:
import sys
import csv
import json
from collections import Counter, defaultdict
from pathlib import Path

assert sys.version_info >= (3, 11), "Use a Python 3.11+ kernel."
PROJECT_OVERRIDE = None  # Optional: Path("/path/to/soccer-recruitment")
if PROJECT_OVERRIDE is not None:
    candidates = [Path(PROJECT_OVERRIDE).expanduser()]
else:
    candidates = []
    for folder in [Path.cwd(), *Path.cwd().parents]:
        candidates.extend([folder, folder / "soccer-recruitment"])
ROOT = next((p.resolve() for p in candidates if (p / "src/soccer/cli.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Start Jupyter from the repository root or notebooks directory, or set PROJECT_OVERRIDE to your complete checkout.")
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
for folder in ("reports", "data/processed"):
    (ROOT / folder).mkdir(parents=True, exist_ok=True)
print("Python:", sys.version.split()[0])
print("Project:", ROOT.name)

paths = sorted((ROOT / "data/raw/statsbomb/events").glob("*.json"))
if not paths:
    raise FileNotFoundError("Run notebook 01 with acquisition enabled first.")
print("Downloaded event matches:", len(paths))


## 1. Settings and downloads

Run cells in order. First execution can download hundreds of MB and take several minutes. We reuse existing 2022 events and fetch 2018 events into a **separate compressed training cache**. Every training file is pinned to the main project's StatsBomb source revision and verified by original-byte SHA-256.

Defaults require both complete 64-match tournaments. `RUN_DOWNLOADS=False` enables offline reruns once data is cached. `ALLOW_PARTIAL_COHORTS=True` is only for a development smoke test: it is saved in the run manifest and does not support a full-tournament conclusion.

The 12×8 grid is a coarse starting choice. The 20-action support threshold only flags sparse zones; it does not make them reliable. Do not choose grid settings by whichever produces a preferred 2022 ranking.


In [ ]:
import sys
import gzip
import hashlib
from datetime import datetime, timezone
sys.path.insert(0,str(ROOT / "src"))
from soccer.acquire import fetch, get

RUN_DOWNLOADS = True
ALLOW_PARTIAL_COHORTS = False
TRAIN_MATCH_LIMIT = 64
EVAL_MATCH_LIMIT = 64
NX, NY = 12, 8
MIN_ZONE_ACTIONS = 20
MIN_TOURNAMENT_MINUTES = 300
MIN_MATCH_MINUTES = 30
TRAIN_ROOT = ROOT / "data/xt_training_2018"
TRAIN_ROOT.mkdir(parents=True,exist_ok=True)
source_lock=json.loads((ROOT / "sources.lock.json").read_text(encoding="utf-8"))
REVISION=source_lock["revisions"]["statsbomb"]
train_lock_path=TRAIN_ROOT / "sources.lock.json"
training_lock=json.loads(train_lock_path.read_text(encoding="utf-8")) if train_lock_path.exists() else dict(revision=REVISION,files={})
if training_lock["revision"] != REVISION:
    raise ValueError("Training cache uses another source revision; use a new cache directory.")

def training_file(remote):
    path=TRAIN_ROOT / (remote+".gz")
    recorded=training_lock["files"].get(remote)
    if path.exists() and recorded:
        with gzip.open(path,"rb") as stream:
            data=stream.read()
        if hashlib.sha256(data).hexdigest()!=recorded["sha256"]:
            raise ValueError(f"Training cache checksum mismatch: {path}")
        return json.loads(data)
    if not RUN_DOWNLOADS:
        raise FileNotFoundError(f"Training data missing: {path}. Enable RUN_DOWNLOADS.")
    url=f"https://raw.githubusercontent.com/statsbomb/open-data/{REVISION}/{remote}"
    data=get(url)
    parsed=json.loads(data)
    path.parent.mkdir(parents=True,exist_ok=True)
    temporary=path.with_suffix(path.suffix+".part")
    temporary.write_bytes(gzip.compress(data,compresslevel=5,mtime=0))
    temporary.replace(path)
    training_lock["files"][remote]=dict(url=url,sha256=hashlib.sha256(data).hexdigest(),bytes=len(data),downloaded_at=datetime.now(timezone.utc).isoformat())
    train_lock_path.write_text(json.dumps(training_lock,indent=2)+"\n",encoding="utf-8")
    print("Training download:",remote,flush=True)
    return parsed

if RUN_DOWNLOADS:
    fetch(ROOT,competition=43,season=106,match_count=EVAL_MATCH_LIMIT,game=1)
train_catalog=training_file("data/matches/43/3.json")
eval_catalog=json.loads((ROOT / "data/raw/statsbomb/matches/43_106.json").read_text(encoding="utf-8"))
train_matches=sorted(train_catalog,key=lambda m:(m["match_date"],m["match_id"]))[:TRAIN_MATCH_LIMIT]
eval_matches=sorted(eval_catalog,key=lambda m:(m["match_date"],m["match_id"]))[:EVAL_MATCH_LIMIT]
if not ALLOW_PARTIAL_COHORTS and (len(train_matches)!=64 or len(eval_matches)!=64):
    raise ValueError("Complete 64-match cohorts required by default.")
assert {m["match_id"] for m in train_matches}.isdisjoint({m["match_id"] for m in eval_matches})
assert max(m["match_date"] for m in train_matches)<min(m["match_date"] for m in eval_matches)
print("Training:",len(train_matches),"matches in 2018 | Application:",len(eval_matches),"matches in 2022")


## 2. Define the actions before fitting

We include passes, observed carries and non-penalty shots **only in possessions labeled Regular Play or From Counter**. Entire restart-origin possessions are excluded, even later movements within those possessions. This is a restrictive scope, not every action with the ball in motion. Shots displayed separately later include all non-penalty situations.

StatsBomb event coordinates are standardized to 120×80 with the acting team attacking toward increasing x; do not flip them again at halftime. We validate bounds and preserve the provider's downward y direction. Boundary x=120/y=80 falls into the last cell. Coordinates are not meters.

All pass attempts enter fitting. Only passes without `pass.outcome` are completed transitions; failed passes terminate with value zero in this simplified process. Carries are treated as observed successful displacements, not successful take-ons. Dribble events, miscontrols, dispossessions and defensive actions are not directly modeled. Missed shots terminate; rebounds are not followed. Invalid coordinates and duplicate IDs are audited.


In [ ]:
import math
from collections import Counter,defaultdict
ALLOWED_PATTERNS={"Regular Play","From Counter"}

def zone(loc,nx=NX,ny=NY):
    if not isinstance(loc,(list,tuple)) or len(loc)<2:
        return None
    x,y=loc[:2]
    if not all(isinstance(v,(int,float)) and not isinstance(v,bool) and math.isfinite(v) for v in (x,y)):
        return None
    if not (0<=x<=120 and 0<=y<=80):
        return None
    return min(ny-1,int(y/80*ny))*nx+min(nx-1,int(x/120*nx))

def extract_actions(events,match_id,nx=NX,ny=NY):
    audit=Counter(events=len(events))
    ids=[e["id"] for e in events]
    if len(ids)!=len(set(ids)):
        raise ValueError(f"Duplicate event IDs: {match_id}")
    actions=[]
    for e in events:
        kind=e["type"]["name"]
        if e["period"]==5 or kind not in {"Pass","Carry","Shot"}:
            continue
        if e.get("play_pattern",{}).get("name") not in ALLOWED_PATTERNS:
            audit["excluded_pattern"]+=1
            continue
        if kind=="Shot" and e["shot"]["type"]["name"]=="Penalty":
            audit["excluded_penalty"]+=1
            continue
        audit["candidate_actions"]+=1
        start=e.get("location")
        start_zone=zone(start,nx,ny)
        if start_zone is None or "player" not in e:
            audit["invalid_start_or_actor"]+=1
            continue
        successful=kind=="Carry" or (kind=="Pass" and "outcome" not in e["pass"])
        end=e.get(kind.lower(),{}).get("end_location") if kind!="Shot" else None
        end_zone=zone(end,nx,ny) if successful else None
        if successful and end_zone is None:
            audit["invalid_successful_end"]+=1
            continue
        actions.append(dict(match_id=str(match_id),event_id=e["id"],event_index=e.get("index"),period=e["period"],timestamp=e["timestamp"],
            team_id=e["team"]["id"],team=e["team"]["name"],player_id=e["player"]["id"],player=e["player"]["name"],
            kind=kind,start_zone=start_zone,end_zone=end_zone,start_x=start[0],start_y=start[1],
            end_x=end[0] if successful else None,end_y=end[1] if successful else None,successful=successful,
            goal=kind=="Shot" and e["shot"]["outcome"]["name"]=="Goal"))
        audit["retained_actions"]+=1
    return actions,dict(audit)

def fit_xt(actions,nx=NX,ny=NY,tolerance=1e-10,max_iterations=10000):
    size=nx*ny
    starts=[0]*size
    goals=[0]*size
    shots=[0]*size
    failed_passes=[0]*size
    transitions=[Counter() for _ in range(size)]
    for a in actions:
        i=a["start_zone"]
        starts[i]+=1
        if a["kind"]=="Shot":
            shots[i]+=1
            goals[i]+=int(a["goal"])
        elif a["successful"]:
            transitions[i][a["end_zone"]]+=1
        else:
            failed_passes[i]+=1
    if not any(goals):
        raise ValueError("Training actions contain no goals; cannot fit a meaningful goal-based surface.")
    values=[0.0]*size
    for iteration in range(1,max_iterations+1):
        updated=[(goals[i]+sum(count*values[j] for j,count in transitions[i].items()))/starts[i] if starts[i] else 0.0 for i in range(size)]
        residual=max(abs(a-b) for a,b in zip(updated,values))
        values=updated
        if residual<tolerance:
            break
    else:
        raise RuntimeError("xT iteration did not converge; inspect transitions and settings.")
    assert all(0<=v<=1+1e-9 for v in values)
    return dict(nx=nx,ny=ny,values=values,start_counts=starts,goals=goals,shot_counts=shots,failed_passes=failed_passes,
                transition_counts=[dict(c) for c in transitions],iterations=iteration,residual=residual)

def rate_movement(action,model):
    if action["kind"] not in {"Pass","Carry"} or not action["successful"]:
        return None
    i,j=action["start_zone"],action["end_zone"]
    if not model["start_counts"][i] or not model["start_counts"][j]:
        return None  # Unsupported zones are missing, not false zero-valued estimates.
    return model["values"][j]-model["values"][i]


## 3. Test the mechanics with a known two-zone example

In this synthetic example, zone 1 has one goal and one missed shot, so its value is 0.5. Zone 0 has one completed pass to zone 1 and one failed pass, so its value is 0.25. A successful move from 0 to 1 adds 0.25; the reverse loses 0.25. This test checks the calculation rather than football validity.


In [ ]:
def toy(kind,start,end=None,successful=False,goal=False):
    return dict(kind=kind,start_zone=start,end_zone=end,successful=successful,goal=goal)
toy_model=fit_xt([toy("Pass",0,1,True),toy("Pass",0),toy("Shot",1,goal=True),toy("Shot",1)],nx=2,ny=1)
assert math.isclose(toy_model["values"][1],0.5)
assert math.isclose(toy_model["values"][0],0.25)
assert math.isclose(rate_movement(toy("Pass",0,1,True),toy_model),0.25)
assert math.isclose(rate_movement(toy("Carry",1,0,True),toy_model),-0.25)
assert rate_movement(toy("Pass",0),toy_model) is None
assert zone([120,80],2,1)==1 and zone([-1,40],2,1) is None
assert zone([float("nan"),40],2,1) is None
empty_model=dict(toy_model,start_counts=[0,2])
assert rate_movement(toy("Pass",0,1,True),empty_model) is None
print("Known-value, failed-move, signed-value, unsupported-zone and boundary checks passed.")

# Filtering and coordinate failures must be visible in extraction diagnostics.
example=dict(id="example", index=1, period=1, timestamp="00:01:00.000", type={"name":"Pass"},
             play_pattern={"name":"Regular Play"}, team={"id":1,"name":"Team"},
             player={"id":1,"name":"Player"}, location=[60,40], **{"pass":{"end_location":[90,40]}})
kept,audit=extract_actions([example],"test")
assert len(kept)==1 and kept[0]["successful"]
failed=dict(example,id="failed",**{"pass":{"outcome":{"name":"Incomplete"},"end_location":[130,40]}})
kept,audit=extract_actions([failed],"test")
assert len(kept)==1 and not kept[0]["successful"]  # Failed end position is not a successful transition.
invalid=dict(example,id="invalid",**{"pass":{"end_location":[130,40]}})
kept,audit=extract_actions([invalid],"test")
assert not kept and audit["invalid_successful_end"]==1
restart=dict(example,id="restart",play_pattern={"name":"From Corner"})
kept,audit=extract_actions([restart],"test")
assert not kept and audit["excluded_pattern"]==1
print("Extraction, restart-scope and coordinate-diagnostic checks passed.")


## 4. Fit on 2018 only

For zone i, the update is:

`V[i] = goals[i] / selected_actions[i] + sum(successful_moves[i,j] / selected_actions[i] * V[j])`

Start from zero and repeat until the largest cell change is below tolerance. Goals here are actual training outcomes, not provider xG. This is a simplified Markov scoring model with failed movements and missed shots absorbing at zero. The surface describes the selected action process; it is not a calibrated probability for every real-world possession.

Unobserved training cells have computational value zero, but movements involving them will be marked unscorable at application time. Low-support cells remain flagged. We do not smooth or tune against 2022 results.


In [ ]:
train_actions=[]
training_audits={}
for match in train_matches:
    mid=str(match["match_id"])
    events=training_file(f"data/events/{mid}.json")
    extracted,audit=extract_actions(events,mid)
    train_actions.extend(extracted)
    training_audits[mid]=audit
model=fit_xt(train_actions)
model["training_match_ids"]=[str(m["match_id"]) for m in train_matches]
model["source_revision"]=REVISION
print("Training actions:",len(train_actions),"| goals:",sum(model["goals"]),"| iterations:",model["iterations"])
print("Unobserved cells:",sum(n==0 for n in model["start_counts"]))
print(f"Cells with fewer than {MIN_ZONE_ACTIONS} actions:",sum(n<MIN_ZONE_ACTIONS for n in model["start_counts"]))
print("The model is now frozen. No 2022 events enter fitting.")


## 5. Audit 2022 minutes and score observed movements

The copied minutes function is the corrected version: bench discipline cannot create minutes. Elapsed participation includes stoppage time and temporary absences. We retain all players, including goalkeepers, and record action coverage. Same-zone moves receive zero; signed negative values are preserved. Failed passes are reported separately and do not receive an action xT value.

This successful-movement attribution can favor safe passers: **net movement xT is not a turnover-adjusted total**. The failure-aware fitting surface does not make action attribution failure-aware. That would be a separately specified model extension.


In [ ]:
def seconds(timestamp):
    h, m, s = timestamp.split(":")
    return int(h)*3600 + int(m)*60 + float(s)

def playing_time(events, roster_ids):
    played_periods = sorted({e["period"] for e in events if 1 <= e["period"] <= 4})
    if played_periods != list(range(1, max(played_periods)+1)):
        raise ValueError("Missing played period")
    durations = {}
    offsets = {}
    for period in played_periods:
        ends = [seconds(e["timestamp"]) for e in events if e["period"] == period and e["type"]["name"] == "Half End"]
        if not ends:
            raise ValueError(f"Period {period} has no Half End")
        offsets[period] = sum(durations.values())
        durations[period] = max(ends)
    total = sum(durations.values())
    active, rows, intervals = {}, {}, []
    removed = set()
    temporary_off = Counter()
    red_cards = Counter()
    permanent_off = Counter()
    bench_discipline_events = 0

    def ensure(team, player):
        key = (team["id"], player["id"])
        if key not in roster_ids:
            raise ValueError(f"Player absent from lineup roster: {key}")
        rows.setdefault(key, dict(team_id=team["id"], team=team["name"], player_id=player["id"], player=player["name"], minutes=0.0))
        return key

    def close(key, t, reason):
        if key not in active:
            raise ValueError(f"{reason}: player not active: {key}")
        start = active.pop(key)
        if t < start:
            raise ValueError("Negative participation interval")
        rows[key]["minutes"] += (t-start)/60
        intervals.append((key, start, t, reason))
        removed.add(key)

    for e in events:
        if e["type"]["name"] == "Starting XI":
            for slot in e["tactics"]["lineup"]:
                key = ensure(e["team"], slot["player"])
                if key in active:
                    raise ValueError("Duplicate starter")
                active[key] = 0.0
    teams = {key[0] for key in active}
    if len(teams) != 2 or any(sum(k[0] == team for k in active) != 11 for team in teams):
        raise ValueError("Expected exactly two Starting XIs of eleven players")
    ordered = sorted((e for e in events if e["period"] in durations), key=lambda e: (e["period"], seconds(e["timestamp"]), e.get("index", 0)))
    for e in ordered:
        t = offsets[e["period"]] + seconds(e["timestamp"])
        if seconds(e["timestamp"]) > durations[e["period"]] + 0.001:
            raise ValueError("Event occurs after period end")
        kind = e["type"]["name"]
        if kind == "Substitution":
            outgoing = ensure(e["team"], e["player"])
            # A prior permanent-departure event may already have ended this player's interval.
            if outgoing in active:
                close(outgoing, t, "substitution")
            elif outgoing not in removed:
                raise ValueError("Outgoing substitute never active")
            incoming = ensure(e["team"], e["substitution"]["replacement"])
            if incoming in active or incoming in removed:
                raise ValueError("Unsupported substitute re-entry")
            active[incoming] = t
        if kind == "Player Off":
            key = ensure(e["team"], e["player"])
            if e.get("player_off", {}).get("permanent", False):
                permanent_off[key[0]] += 1
                if key in active:
                    close(key, t, "permanent departure")
            else:
                temporary_off[key] += 1
        if kind == "Bad Behaviour" and e.get("position", {}).get("name") == "Substitute":
            bench_discipline_events += 1
            continue  # Bench discipline does not create participation time or an on-field dismissal.
        card = e.get("foul_committed", {}).get("card", {}).get("name") or e.get("bad_behaviour", {}).get("card", {}).get("name")
        if card in ("Red Card", "Second Yellow"):
            key = ensure(e["team"], e["player"])
            if key in active:
                close(key, t, card)
                red_cards[key[0]] += 1
            else:
                raise ValueError(f"Dismissal of inactive player needs manual review: {key}")
    for key in list(active):
        close(key, total, "match end")
    for key, row in rows.items():
        row["temporary_off_events"] = temporary_off[key]
        if not 0 <= row["minutes"] <= total/60 + 1e-8:
            raise ValueError("Invalid minutes")
    audit = dict(match_elapsed_minutes=total/60, period_minutes={str(p):d/60 for p,d in durations.items()},
                 temporary_off_events=sum(temporary_off.values()), dismissals=sum(red_cards.values()),
                 permanent_departures=sum(permanent_off.values()), participants=len(rows), bench_discipline_events=bench_discipline_events)
    for team in teams:
        actual = sum(r["minutes"] for k,r in rows.items() if k[0] == team)
        expected = 11*total/60
        if actual > expected + 1e-6:
            raise ValueError("Team minutes exceed eleven-player maximum")
        if not red_cards[team] and not permanent_off[team] and abs(actual-expected) > 1e-6:
            raise ValueError("Team participation time does not reconcile")
    return rows, audit


In [ ]:
events_by_match={}
minutes_by_match={}
minutes_audits={}
application_audits={}
action_values=[]
match_metadata={str(m["match_id"]):m for m in eval_matches}
player_matches={}
for match in eval_matches:
    mid=str(match["match_id"])
    path=ROOT / f"data/raw/statsbomb/events/{mid}.json"
    events=json.loads(path.read_text(encoding="utf-8"))
    # Source-byte verification prevents quietly using modified evaluation events.
    entry=source_lock["files"].get(f"statsbomb/events/{mid}.json")
    if RUN_DOWNLOADS:
        source_lock=json.loads((ROOT / "sources.lock.json").read_text(encoding="utf-8"))
        entry=source_lock["files"].get(f"statsbomb/events/{mid}.json")
    if entry is None or hashlib.sha256(path.read_bytes()).hexdigest()!=entry["sha256"]:
        raise ValueError(f"Missing or mismatched evaluation provenance: {mid}")
    roster_path=ROOT / f"data/raw/statsbomb/lineups/{mid}.json"
    roster_entry=source_lock["files"].get(f"statsbomb/lineups/{mid}.json")
    if roster_entry is None or hashlib.sha256(roster_path.read_bytes()).hexdigest()!=roster_entry["sha256"]:
        raise ValueError(f"Missing or mismatched lineup provenance: {mid}")
    roster=json.loads(roster_path.read_text(encoding="utf-8"))
    roster_ids={(team["team_id"],p["player_id"]) for team in roster for p in team["lineup"]}
    exposure,minutes_audits[mid]=playing_time(events,roster_ids)
    minutes_by_match[mid]=exposure
    for key,row in exposure.items():
        player_matches[(mid,*key)]=dict(match_id=mid,match_date=match["match_date"],team_id=key[0],team=row["team"],player_id=key[1],player=row["player"],
            minutes=row["minutes"],matches=1,pass_xt=0.0,carry_xt=0.0,movement_xt=0.0,eligible_moves=0,scored_moves=0,unscored_moves=0,
            low_support_moves=0,failed_openplay_passes=0,nonpen_shots=0,npxg=0.0,nonpen_goals=0)
    extracted,application_audits[mid]=extract_actions(events,mid)
    for a in extracted:
        key=(mid,a["team_id"],a["player_id"])
        if key not in player_matches:
            raise ValueError(f"Playing action without participation interval: {mid}, {a['event_id']}")
        row=player_matches[key]
        if a["kind"]=="Pass" and not a["successful"]:
            row["failed_openplay_passes"]+=1
        if a["kind"] not in {"Pass","Carry"} or not a["successful"]:
            continue
        row["eligible_moves"]+=1
        value=rate_movement(a,model)
        low=min(model["start_counts"][a["start_zone"]],model["start_counts"][a["end_zone"]])<MIN_ZONE_ACTIONS
        record=dict(a,xt=value,low_training_support=low)
        action_values.append(record)
        if value is None:
            row["unscored_moves"]+=1
            continue
        row["scored_moves"]+=1
        row["low_support_moves"]+=int(low)
        row["movement_xt"]+=value
        row["pass_xt" if a["kind"]=="Pass" else "carry_xt"]+=value
    for e in events:
        if e["period"]!=5 and e["type"]["name"]=="Shot" and e["shot"]["type"]["name"]!="Penalty":
            key=(mid,e["team"]["id"],e["player"]["id"])
            if key not in player_matches:
                raise ValueError("Shot without participation interval")
            row=player_matches[key]
            row["nonpen_shots"]+=1
            row["npxg"]+=e["shot"].get("statsbomb_xg",0.0)
            row["nonpen_goals"]+=int(e["shot"]["outcome"]["name"]=="Goal")
print("Successful eligible movements:",len(action_values))
print("Unscorable movements:",sum(a["xt"] is None for a in action_values))
print("Scored low-support movements:",sum(a["xt"] is not None and a["low_training_support"] for a in action_values))


## 6. Who contributed most in a game?

Choose a match below. We rank **signed movement xT totals**, showing passing, carrying, and separate shooting statistics. This answers a narrow on-ball question, not player of the match. The displayed list requires at least 30 minutes and no unscorable eligible movements; shorter appearances remain in the saved table. Asterisk means at least one scored movement uses a sparse training zone.


In [ ]:
SELECTED_MATCH_ID=str(eval_matches[0]["match_id"])  # Replace with another analyzed match ID.
if SELECTED_MATCH_ID not in match_metadata:
    raise ValueError("Select an analyzed 2022 match ID.")

def add_rates(row):
    row["movement_xt_per90"]=90*row["movement_xt"]/row["minutes"] if row["minutes"]>0 else None
    row["movement_coverage"]=row["scored_moves"]/row["eligible_moves"] if row["eligible_moves"] else None
    return row
match_rows=[add_rates(dict(r)) for r in player_matches.values()]
match=match_metadata[SELECTED_MATCH_ID]
print(match["home_team"]["home_team_name"],"vs",match["away_team"]["away_team_name"],match["match_date"])
selected=sorted([r for r in match_rows if r["match_id"]==SELECTED_MATCH_ID and r["minutes"]>=MIN_MATCH_MINUTES and r["unscored_moves"]==0 and r["scored_moves"]>0],key=lambda r:(-r["movement_xt"],r["player_id"]))
print("Rank | Player | minutes | passing xT | carrying xT | movement xT | npxG | NP goals")
for rank,r in enumerate(selected[:10],1):
    flag="*" if r["low_support_moves"] else ""
    print(f"{rank:2} | {r['player']}{flag} | {r['minutes']:.1f} | {r['pass_xt']:+.4f} | {r['carry_xt']:+.4f} | {r['movement_xt']:+.4f} | {r['npxg']:.3f} | {r['nonpen_goals']}")


## 7. Tournament totals versus rates

Total contribution and contribution per 90 answer different questions. Teams reaching later rounds offer more opportunities. We apply the predeclared 300-minute filter to the rate view and show minutes. No role filter is imposed; compare CMs, wingers and strikers with their role-specific notebooks before interpreting recruitment suitability. Players with unscorable moves are excluded from the displayed rankings, not erased from the output.


In [ ]:
tournament={}
sum_fields=["minutes","matches","pass_xt","carry_xt","movement_xt","eligible_moves","scored_moves","unscored_moves","low_support_moves","failed_openplay_passes","nonpen_shots","npxg","nonpen_goals"]
for r in match_rows:
    key=(r["team_id"],r["player_id"])
    total=tournament.setdefault(key,dict(team_id=r["team_id"],team=r["team"],player_id=r["player_id"],player=r["player"],**{f:0 for f in sum_fields}))
    for f in sum_fields:
        total[f]+=r[f]
tournament_rows=[add_rates(r) for r in tournament.values()]
rankable=[r for r in tournament_rows if r["unscored_moves"]==0 and r["scored_moves"]>0]
for label,metric,pool in [("Tournament movement xT total","movement_xt",rankable),
                          (f"Movement xT/90, at least {MIN_TOURNAMENT_MINUTES} minutes","movement_xt_per90",[r for r in rankable if r["minutes"]>=MIN_TOURNAMENT_MINUTES])]:
    print("\n"+label)
    ranked=sorted(pool,key=lambda r:(-r[metric],r["player_id"]))
    if not ranked:
        print("No eligible rows. A partial cohort may be insufficient.")
    for r in ranked[:10]:
        print(f"{r['player']} ({r['team']}) | {r[metric]:+.4f} | {r['minutes']:.1f} min | {r['matches']} matches | low-support moves: {r['low_support_moves']}")


## 8. Explain the leading player's actions

The top positive movements show where value came from. Also inspect negative values: recycling the ball can reduce location-based xT while being tactically sensible. A location-only model cannot observe defenders being drawn out of position.


In [ ]:
FOCUS_PLAYER_ID=selected[0]["player_id"] if selected else None  # Change this to inspect another player.
focus_actions=[a for a in action_values if a["match_id"]==SELECTED_MATCH_ID and a["player_id"]==FOCUS_PLAYER_ID and a["xt"] is not None]
for a in sorted(focus_actions,key=lambda a:(-a["xt"],a["event_index"]))[:10]:
    print(f"{a['kind']} | period {a['period']} {a['timestamp']} | ({a['start_x']:.1f},{a['start_y']:.1f}) -> ({a['end_x']:.1f},{a['end_y']:.1f}) | xT {a['xt']:+.4f} | event {a['event_id']}")


## 9. Optional pitch visualizations

The calculations use Python's standard library. For plots, install matplotlib in this notebook's kernel by running `%pip install matplotlib` in a new code cell, then rerun the plotting cell below. If it is unavailable, the notebook still finishes and writes the tables. Charts are saved as PNGs for the portfolio.

Both surfaces use the same grid: x increases toward the attacking goal on the right; y increases downward. Training support is shown beside model values. The action map includes signed positive and negative movements for the selected player/match, not defensive positioning.


In [ ]:
try:
    import matplotlib.pyplot as plt
except ModuleNotFoundError:
    plt=None
    print("Plots skipped. Run %pip install matplotlib in a new cell to enable them.")

if plt is not None:
    plot_dir=ROOT / "reports/figures"
    plot_dir.mkdir(parents=True,exist_ok=True)
    def pitch(ax):
        ax.set_xlim(0,120); ax.set_ylim(80,0); ax.set_aspect("equal")
        ax.plot([0,120,120,0,0],[0,0,80,80,0],color="gray",lw=1)
        ax.plot([60,60],[0,80],color="gray",lw=1)
        for xs in ([0,18,18,0],[120,102,102,120]):
            ax.plot(xs,[18,18,62,62],color="gray",lw=1)
        ax.set_xlabel("StatsBomb x units; attack to the right")
        ax.set_ylabel("StatsBomb y units")
    fig,axes=plt.subplots(1,2,figsize=(13,4),constrained_layout=True)
    for ax,values,title in [(axes[0],model["values"],"2018 fitted xT surface"),(axes[1],model["start_counts"],"2018 selected-action support")]:
        grid=[values[y*NX:(y+1)*NX] for y in range(NY)]
        im=ax.imshow(grid,extent=[0,120,80,0],origin="upper",cmap="YlOrRd",vmin=0)
        pitch(ax); ax.set_title(title); fig.colorbar(im,ax=ax,shrink=.8)
    fig.savefig(plot_dir / "xt_surface_and_support.png",dpi=180)
    plt.show()
    if focus_actions:
        fig,ax=plt.subplots(figsize=(9,6),constrained_layout=True)
        pitch(ax)
        for a in focus_actions:
            color="tab:blue" if a["xt"]>=0 else "tab:orange"
            ax.annotate("",xy=(a["end_x"],a["end_y"]),xytext=(a["start_x"],a["start_y"]),
                        arrowprops=dict(arrowstyle="->",color=color,alpha=.65,lw=.7+min(abs(a["xt"])*35,3)))
        ax.set_title(f"{focus_actions[0]['player']} | match {SELECTED_MATCH_ID}\nBlue: nonnegative xT change; orange: negative")
        fig.savefig(plot_dir / f"xt_actions_{SELECTED_MATCH_ID}_{FOCUS_PLAYER_ID}.png",dpi=180)
        plt.show()


## 10. Reconcile and export

These checks verify attribution arithmetic and chronological separation. They do **not** establish predictive calibration, causal impact or rank stability. A historical fit and a later application are necessary safeguards, not a complete model evaluation.

Outputs include every scored or unscorable eligible successful movement, all player-match rows, tournament summaries, the frozen model, and a run manifest listing match IDs, settings and audit counts. Unscorable values are missing, not zero; displayed rankings require complete eligible-movement coverage.


In [ ]:
assert set(model["training_match_ids"]).isdisjoint(match_metadata)
scored=[a for a in action_values if a["xt"] is not None]
assert math.isclose(sum(a["xt"] for a in scored),sum(r["movement_xt"] for r in match_rows),abs_tol=1e-9)
assert math.isclose(sum(r["movement_xt"] for r in match_rows),sum(r["movement_xt"] for r in tournament_rows),abs_tol=1e-9)
for r in match_rows:
    assert r["eligible_moves"]==r["scored_moves"]+r["unscored_moves"]
    assert math.isclose(r["movement_xt"],r["pass_xt"]+r["carry_xt"],abs_tol=1e-9)
    if r["minutes"]>0:
        assert math.isclose(r["movement_xt_per90"]*r["minutes"]/90,r["movement_xt"],abs_tol=1e-9)
report_dir=ROOT / "reports"
report_dir.mkdir(parents=True,exist_ok=True)
def save_rows(filename,rows):
    if not rows:
        raise ValueError(f"No rows to save: {filename}")
    with (report_dir / filename).open("w",newline="",encoding="utf-8") as stream:
        writer=csv.DictWriter(stream,fieldnames=list(rows[0]))
        writer.writeheader(); writer.writerows(rows)
for filename,rows in [("xt_action_values.csv",action_values),("xt_player_match.csv",match_rows),("xt_tournament_players.csv",tournament_rows)]:
    save_rows(filename,rows)
(report_dir / "xt_model_2018.json").write_text(json.dumps(model,indent=2)+"\n",encoding="utf-8")
manifest=dict(created_at=datetime.now(timezone.utc).isoformat(),python=sys.version,source_revision=REVISION,
    training_match_ids=model["training_match_ids"],application_match_ids=list(match_metadata),grid=[NX,NY],
    allowed_patterns=sorted(ALLOWED_PATTERNS),training_variant="failed passes and missed shots terminal at zero; successful movements rated by signed end-minus-start",
    partial_cohorts_allowed=ALLOW_PARTIAL_COHORTS,min_zone_actions=MIN_ZONE_ACTIONS,min_tournament_minutes=MIN_TOURNAMENT_MINUTES,
    min_match_minutes=MIN_MATCH_MINUTES,training_audits=training_audits,application_audits=application_audits,minutes_audits=minutes_audits,
    scored_movements=len(scored),unscorable_movements=len(action_values)-len(scored),
    low_support_scored_movements=sum(a["low_training_support"] for a in scored),plots_generated=plt is not None)
(report_dir / "xt_run_manifest.json").write_text(json.dumps(manifest,indent=2)+"\n",encoding="utf-8")
print("Reconciliation passed. Outputs saved in",report_dir)


## What the portfolio can claim

“Implemented an auditable location-based possession-value baseline, trained on historical events and applied to a later tournament, with player-match attribution, exposure-aware summaries, sparse-zone diagnostics and action maps.”

Avoid claiming measured overall match impact, validated player ability, discovered undervalued professionals, or calibrated scoring probabilities. This model values locations, not the difficulty or context of getting there. Event-derived carry definitions and failed-action omissions can affect rankings. International tournaments and unequal team progression create selection/exposure effects.

**Next analysis:** inspect the largest actions and sparse cells, compare role-specific peers, and report uncertainty/rank sensitivity using independent matches. Predeclare training-only grid/smoothing experiments or use a separate intermediate validation cohort; do not tune on 2022 and continue calling it untouched. A more complete on-ball model such as VAEP would model scoring and conceding probabilities and value more action types, but requires additional data and proper evaluation.
